#### Set Up

In [1]:
import os
import json
from tqdm import tqdm
from dotenv import load_dotenv

import warnings
warnings.filterwarnings('ignore')

import torch

from selfcheck_prompt_api import SelfCheckPromptAPI

In [2]:
load_dotenv()

openai_api_key = os.getenv('OPENAI_API_KEY')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
selfcheck_prompt = SelfCheckPromptAPI(
    model='gpt-4o-mini-2024-07-18',
    api_key=openai_api_key,
    prompt_template_path='selfcheck_prompt_template.txt'
)

#### Dataset

In [4]:
with open("data/dataset.json", "r") as f:
    dataset = json.loads(f.read())

print(f"The length of the dataset: {len(dataset)}")

The length of the dataset: 238


In [5]:
print("The keys of each sample:", list(dataset[0].keys()))

The keys of each sample: ['gpt3_text', 'wiki_bio_text', 'gpt3_sentences', 'annotation', 'wiki_bio_test_idx', 'gpt3_text_samples']


#### Benchmark

In [6]:
hallucination_scores = {}

for i in tqdm(range(len(dataset[:1]))):
    sample = dataset[i]
    idx = sample['wiki_bio_test_idx']
    
    hallucination_scores[idx] = selfcheck_prompt.predict_hallucination(
        sentences=sample['gpt3_sentences'],
        sample_responses=sample['gpt3_text_samples'],
        verbose=True
    )

100%|██████████| 1/1 [02:34<00:00, 154.40s/it]


In [8]:
with open("data/scores_gpt4o_mini.json", "w") as file:
    json.dump(hallucination_scores, file)